# Chapter 3 — Databases (PostgreSQL)
### ML Engineer (Production) Interview Playbook

**Topics:** SQL · JOIN · Index · Transaction · ACID · Isolation · Lock · PostgreSQL

> A deep understanding of relational databases is critical for a production role, especially at
> financial companies like bunq, where data correctness, concurrency, and durability are critical. A
> single mistake in transaction handling can lead to money being deducted twice or a negative balance.
> This chapter focuses on PostgreSQL, but the concepts generally transfer. A tech lead expects you not
> just to write SQL, but to know what's happening under the hood and where the trade-offs are.
>
> **Interview tip:** The financial mindset: in this domain, "almost correct" means wrong. Every
> operation that moves money must be atomic, isolated, and recoverable. Show that same mindset in your
> answers.

## 3.1 SQL and the Relational Model

The relational model organizes data into tables of rows and columns, linked by keys. A **primary key**
makes each row unique, and a **foreign key** enforces the relationship between tables and referential
integrity.

In [ ]:
-- accounts.sql
CREATE TABLE accounts (
    id BIGINT PRIMARY KEY,
    owner_id BIGINT NOT NULL REFERENCES users(id),
    balance NUMERIC(15, 2) NOT NULL DEFAULT 0,   -- money: NUMERIC, not float
    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    CONSTRAINT positive_balance CHECK (balance >= 0)   -- database-level constraint
);


**Note:** Constraints (`CHECK`, `NOT NULL`, `UNIQUE`, `FK`) are the last line of defense for data
correctness, enforced at the database level — even if an application bug bypasses them. In a financial
system, relying only on app-side validation is not enough.

### The logical execution order of a query

Contrary to how it's written, the database executes a query in this logical order; knowing this
explains a lot of behavior (e.g. why you can't use a `SELECT` column alias in `WHERE`):

- `FROM`/`JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `DISTINCT` → `ORDER BY` → `LIMIT`

**Interview tip:** Because `WHERE` runs before `SELECT`, aliases defined in `SELECT` don't exist yet
and can't be used in `WHERE` (but they can in `ORDER BY`, which runs after `SELECT`). A classic
question.

### Q3.1 — What's the difference between `WHERE` and `HAVING`?

`WHERE` filters rows before grouping and operates on raw columns. `HAVING` runs after `GROUP BY` and
filters on the result of aggregate functions (like `COUNT`, `SUM`). Example: `WHERE amount > 0` keeps
valid rows, then `GROUP BY user_id` groups them, and `HAVING SUM(amount) > 1000` returns only users
whose total exceeds one thousand. You cannot put an aggregate condition in `WHERE`.

## 3.2 JOIN

`JOIN` combines data from multiple tables based on a condition. Choosing the right join type matters —
it determines both the result and performance.

| Join type | What it returns |
|---|---|
| `INNER JOIN` | Only rows that match in both tables |
| `LEFT JOIN` | All rows from the left table + matches from the right (or `NULL`) |
| `RIGHT JOIN` | All rows from the right table + matches from the left (or `NULL`) |
| `FULL OUTER JOIN` | All rows from both sides, with `NULL` where there's no match |
| `CROSS JOIN` | Cartesian product (every row with every row) |

In [ ]:
-- Total transactions per user, including users with no transactions
SELECT u.id, u.name, COALESCE(SUM(t.amount), 0) AS total
FROM users u
LEFT JOIN transactions t ON t.user_id = u.id
GROUP BY u.id, u.name
ORDER BY total DESC;


**Interview tip:** Why `LEFT JOIN` and `COALESCE`? An `INNER JOIN` would drop users with no
transactions at all. `LEFT JOIN` keeps them, and `COALESCE` turns the resulting `NULL` into zero.
Recognizing when you need `LEFT` vs. `INNER` is a very common question.

### Q3.2 — A user exists in the `users` table but has no transactions in `transactions`. How do you get all users with their transaction count (even zero)?

With a `LEFT JOIN` from `users` to `transactions` and `COUNT` on the transaction column (not `*`).
Subtle point: use `COUNT(t.id)` instead of `COUNT(*)`, because `COUNT(*)` counts the rows produced by
the `LEFT JOIN` — which is one row with `NULL` for a user with no transactions, wrongly giving 1. But
`COUNT(t.id)` doesn't count `NULL` values, correctly giving 0.

In [ ]:
SELECT u.id, COUNT(t.id) AS tx_count
FROM users u
LEFT JOIN transactions t ON t.user_id = u.id
GROUP BY u.id;


## 3.3 Index

An index is a data structure (B-Tree by default) that brings lookups from a full table scan (O(n)) down
to O(log n). Without a proper index, queries on a large table become slow. But an index isn't free:
every `INSERT`/`UPDATE`/`DELETE` must also update the index, and it consumes space.

In [ ]:
-- Simple index on a frequently queried column
CREATE INDEX idx_tx_user_id ON transactions(user_id);

-- Composite index -- column order is critical
CREATE INDEX idx_tx_user_time ON transactions(user_id, created_at DESC);

-- Partial index: covers only a subset of rows
CREATE INDEX idx_pending ON orders(created_at) WHERE status = 'pending';

-- Unique index: both speed and a uniqueness guarantee (e.g. for an idempotency key)
CREATE UNIQUE INDEX idx_idem ON payments(idempotency_key);


### Column order in a composite index (leftmost prefix)

An index on `(a, b)` is useful for queries that filter on `a` or on `(a AND b)`, but is useless for a
query that filters only on `b`. Like a phone book sorted by last name then first name: searching by
last name is fast, but searching only by first name gets no help.

### Index types in PostgreSQL

| Type | Use case |
|---|---|
| B-Tree | Default; equality and range (`=`, `<`, `>`, `BETWEEN`, `ORDER BY`) |
| Hash | Equality only (`=`) |
| GIN | Composite data: `JSONB`, arrays, full-text search |
| GiST / SP-GiST | Spatial and range data |
| BRIN | Very large, naturally ordered tables (e.g. time-series data) |

### Checking index usage with `EXPLAIN`

In [ ]:
EXPLAIN ANALYZE
SELECT * FROM transactions WHERE user_id = 42;

-- "Seq Scan" means the index was not used (probably doesn't exist)
-- "Index Scan" means the index was correctly used


**Note:** The golden question: "Is an index always good?" No. On a low-cardinality column (e.g.
a boolean, or a column with only 2-3 values), a B-Tree index is useless because it covers a large
fraction of the table and the planner prefers a seq scan. Also, every index slows down writes and
consumes space. Index write-heavy tables with caution.

### Q3.3 — A query on a 100-million-row table is slow. How do you diagnose and fix it?

First I check the execution plan with `EXPLAIN ANALYZE` to see where time goes and whether a `Seq Scan`
is happening. If the filtered column has no index, I create an appropriate one (often composite, with
column order matching the query pattern). If the query only needs a few columns, a covering index
(with `INCLUDE`) can enable an index-only scan and avoid hitting the table at all. I also check whether
table statistics are up to date (`ANALYZE`), since the planner relies on them. For very large tables,
partitioning can also shrink the scan's scope.

## 3.4 Transactions and ACID

A transaction is a set of operations executed as an indivisible unit: either they all succeed and
commit together, or an error rolls all of them back. ACID is the set of four guarantees a relational
database gives for a transaction:

| Letter | Guarantee | What it means in practice |
|---|---|---|
| A — Atomicity | All or nothing | On error, the entire transaction is rolled back |
| C — Consistency | Constraints preserved | Moves from one valid state to another (FK, CHECK are enforced) |
| I — Isolation | Concurrency separation | Parallel transactions don't produce unwanted interference |
| D — Durability | Durability | After commit, data survives even a power outage |

In [ ]:
-- Money transfer: must be atomic
BEGIN;

UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 2;

COMMIT;

-- If an error (or power loss) occurs between the two UPDATEs, ROLLBACK reverts both
-- We never end up with 100 deducted but not credited


**Interview tip:** How does Durability work? Before confirming a commit, the database writes the
change to the Write-Ahead Log (WAL) on disk. Even if power is lost immediately after, the change is
recovered from the WAL on restart. Know this WAL concept for Postgres.

### Q3.4 — Explain ACID with a real-world example.

Transferring €100 between two accounts: Atomicity guarantees either both `UPDATE`s (debit and credit)
happen or neither does — money never vanishes or duplicates. Consistency guarantees constraints (like
a non-negative balance) aren't violated. Isolation guarantees that if another transaction runs
concurrently on the same account, the result isn't incorrect (e.g. two simultaneous withdrawals don't
push the balance negative). Durability guarantees that after the transfer is confirmed, it stays
recorded even if the server crashes.

## 3.5 Isolation Levels and MVCC

Isolation levels determine how isolated concurrent transactions are from each other. This is a classic
trade-off: stronger isolation means more correctness but less concurrency (and performance). To
understand the levels, first know three unwanted phenomena:

- **Dirty Read** — reading data that another transaction hasn't committed yet (and might roll back).
- **Non-repeatable Read** — reading the same row twice within one transaction and getting different
  values (because another transaction changed it in between).
- **Phantom Read** — running the same conditional query twice and getting a different row count
  (because a new row matching the condition was added).

| Isolation level | Dirty | Non-repeatable | Phantom |
|---|---|---|---|
| Read Uncommitted | Possible* | Possible | Possible |
| Read Committed | No | Possible | Possible |
| Repeatable Read | No | No | Possible* |
| Serializable | No | No | No |

**Note:** PostgreSQL specifics: (1) In Postgres, even Read Uncommitted behaves like Read Committed, and
a Dirty Read never actually happens. (2) In Postgres, Repeatable Read is stronger than the standard and
effectively prevents Phantom Reads too. Postgres's default is Read Committed.

### MVCC — how Postgres achieves concurrency

MVCC (Multi-Version Concurrency Control) is a mechanism that, instead of locking for reads, keeps
multiple versions of each row. Every transaction sees a consistent "snapshot" of the data. Key result:
readers don't block writers, and vice versa. Why does this matter? Because in a high-traffic system,
read locks become a bottleneck; MVCC solves this.

**Interview tip:** Serializable in Postgres is implemented via SSI (Serializable Snapshot Isolation)
and works without heavy locking, but it may reject a transaction with a "serialization failure" at
commit time. So your code must be ready to retry the transaction. This is an important practical point.

### Q3.5 — What's Postgres's default isolation level, and when do you raise it?

The default is Read Committed, which is sufficient for most work: each statement sees the latest
committed data and there's no Dirty Read. I raise it to Repeatable Read when a transaction reads
multiple times and all reads need to be consistent (e.g. generating a consistent report). I go to
Serializable when business logic needs the full guarantee that transactions behave as if executed one
after another (e.g. checking a condition and then writing based on it, where a race condition is
dangerous). The cost is reduced concurrency and the need to retry on serialization failure.

## 3.6 Locking and Concurrency

Locks prevent concurrent interference. There are two main approaches to handling concurrent updates,
and their difference is the most frequently asked concurrency question:

| Approach | How it works | Best for |
|---|---|---|
| Pessimistic Lock | Locks the row (`SELECT ... FOR UPDATE`) so no one else can change it concurrently | High contention |
| Optimistic Lock | Doesn't lock; detects conflict at write time via a `version` column | Low contention |

### Pessimistic locking: safely updating a balance

In [ ]:
BEGIN;

-- Locks the row until the transaction ends; other transactions wait
SELECT balance FROM accounts WHERE id = 1 FOR UPDATE;

-- Now safely compute and update
UPDATE accounts SET balance = balance - 100 WHERE id = 1;

COMMIT;


### Optimistic locking: using a `version` column

In [ ]:
-- The UPDATE only succeeds if version hasn't changed
UPDATE accounts
SET balance = balance - 100, version = version + 1
WHERE id = 1 AND version = 7;

-- If rows affected = 0, someone else updated first -> retry


**Note:** A deadlock happens when two transactions lock each other's resources in reverse order
(transaction A holds lock X and waits for Y; B holds lock Y and waits for X). Postgres detects this and
aborts one with an error. Prevention: always lock resources in a fixed order (e.g. ascending id).

**Practical tip for a job queue:** to safely pick up a job from a queue table without two workers
grabbing the same job, use `SELECT ... FOR UPDATE SKIP LOCKED` — it skips already-locked rows so the
next worker moves on to the next free job.

### Q3.6 — Two concurrent requests want to withdraw from the same account. How do you prevent a negative balance?

Several approaches: (1) Pessimistic locking with `SELECT ... FOR UPDATE` on the account row inside the
transaction, so the second withdrawal waits and sees the updated balance. (2) Optimistic locking with a
`version` column and retry on conflict. (3) As a final line of defense, a `CHECK (balance >= 0)`
constraint at the database level, so even if app logic is wrong, an `UPDATE` that would push it negative
is rejected. In financial systems, a combination of `FOR UPDATE` for correctness and `CHECK` as the
final guarantee is common. It's important to recognize that reading a balance without a lock and then
writing is a classic race condition.

## 3.7 PostgreSQL in Production

Points that make the difference between a stable database and a troublesome one:

- **Connection Pooling:** opening every connection is expensive. Use a pool (e.g. PgBouncer);
  transaction pooling mode gives the best connection efficiency.
- **JSONB:** for semi-structured data; unlike text-based JSON, it's binary and indexable (with GIN).
- **Partitioning:** split a large table into smaller partitions (e.g. by month) for faster queries and
  maintenance.
- **VACUUM / autovacuum:** since MVCC keeps old row versions (dead tuples), `VACUUM` reclaims space and
  prevents bloat and the transaction ID wraparound problem.
- **Read Replica:** read-only copies to distribute read load (e.g. reporting) and reduce pressure on
  the primary node.

**Note:** Because Postgres's MVCC keeps old versions, heavy `UPDATE`/`DELETE` traffic causes dead
tuples to accumulate — "table bloat." If autovacuum isn't tuned correctly, the table grows large and
slow. This is a common Postgres operational issue worth knowing.

### Q3.7 — Why does connection pooling matter for an async service?

Every Postgres connection consumes a server-side process and some memory, and opening/closing one is
expensive. An async service with high concurrency can have hundreds of simultaneous requests; if each
opened its own connection, the database would exceed its connection limit and run out of resources. A
pool keeps a limited number of connections and shares them across requests. Tools like PgBouncer
(external) or a driver's built-in pool (e.g. an asyncpg pool) manage this. Subtle point: in transaction
pooling mode, you can't rely on session-level features like a long-lived prepared statement.

## 3.8 Common Patterns and Anti-Patterns

### The N+1 query problem

The most common performance anti-pattern: a query fetches a list of N entities, then a separate query
runs for each one (1 + N queries). Extremely common with ORMs (lazy loading).

In [ ]:
# Anti-pattern: N+1
users = repo.all()               # 1 query
for u in users:
    print(u.account.balance)     # one query each time -> N queries

# Fix: eager loading / JOIN
users = repo.all_with_accounts() # 1 query with a JOIN
# or in SQLAlchemy: .options(selectinload(User.account))


### Bulk insert/update and upsert

Instead of a thousand separate `INSERT`s, use a bulk operation. For "insert, or update if it already
exists," use `ON CONFLICT` — which is also great for idempotency.

In [ ]:
-- Atomic upsert; update if the unique key already exists
INSERT INTO daily_scores (user_id, day, score)
VALUES (42, '2026-01-10', 0.9)
ON CONFLICT (user_id, day)
DO UPDATE SET score = EXCLUDED.score;


### Keyset pagination (a reminder)

For paginating large tables, instead of `OFFSET` (which gets slower with each page because it reads
and discards all prior rows), use keyset pagination.

In [ ]:
-- Instead of OFFSET 100000 LIMIT 20 (slow):
SELECT * FROM transactions
WHERE id > :last_seen_id
ORDER BY id
LIMIT 20;   -- fast and consistent even on deep pages


### Q3.8 — What is N+1, and how do you detect and fix it in an ORM?

N+1 is when, for a list of N items, N+1 queries run instead of one — usually because the ORM lazily
loads relationships, and accessing each relationship fires a query. Detection: by enabling query
echo/logging or APM tools, I look at the query count for a single request; if the query count grows
linearly with the data size, I have an N+1. Fix: eager loading (e.g. `selectinload` or `joinedload` in
SQLAlchemy) or writing an explicit `JOIN` so related data arrives in one (or a few) queries.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **SQL:** Logical order `FROM→WHERE→GROUP BY→HAVING→SELECT→ORDER BY`; `WHERE` before aggregation,
  `HAVING` after; money as `NUMERIC` with a `CHECK` constraint.
- **JOIN:** `LEFT` to keep unmatched rows; `COUNT(col)` not `COUNT(*)` with a `LEFT JOIN`.
- **Index:** B-Tree default, O(log n); composite column order matters (leftmost prefix); useless on
  low-cardinality columns; `EXPLAIN ANALYZE` to verify.
- **ACID:** Atomicity (all/nothing), Consistency (constraints), Isolation (concurrency), Durability
  (WAL).
- **Isolation:** Dirty/Non-repeatable/Phantom reads; Postgres default = Read Committed; Postgres never
  gives a Dirty Read; MVCC = readers and writers don't block each other; Serializable may need a retry.
- **Locking:** Pessimistic (`FOR UPDATE`) for high contention; Optimistic (`version`) for low
  contention; prevent deadlocks with a fixed lock order; `SKIP LOCKED` for queues.
- **PostgreSQL:** connection pool (PgBouncer); JSONB with GIN; partitioning; VACUUM for dead
  tuples/bloat; read replicas.
- **Patterns:** avoid N+1 with eager loading; bulk/upsert with `ON CONFLICT`; keyset pagination instead
  of `OFFSET`.